# Detecting Reward Hacking in Agent Trajectories

## 0. Overview

Reward hacking is when a coding agent exploits ambiguities or flaws in its
reward signal to score well without actually solving the task — modifying a
unit test so it passes, hardcoding an expected output, bypassing runtime
constraints. Detecting this behaviour automatically is an open problem in AI
safety.

This notebook frames detection as **unsupervised anomaly detection**. The
premise is that benign coding-agent behaviour has a learnable latent
structure, and hacked behaviour deviates from that structure in detectable
ways. Instead of a supervised classifier (which would require labelled hacks
at training time, of which very few exist), we train only on *benign*
trajectories and score evaluation trajectories by how well they fit the
learned distribution of normal behaviour.

### Research question

> *Can unsupervised models trained exclusively on benign coding-agent
> trajectories learn a representation of normal behaviour from which reward
> hacking deviates in a geometrically consistent and detectable way, and does
> this hold across datasets with different task types, hack categories, and
> levels of hack subtlety?*

### What this notebook does

The notebook is organised around a single pipeline that runs end to end:

1. **Data grabbing and preprocessing** — load (or auto-build, on first run)
   three normalized datasets (SWE-smith for training, TRACE and MALT for
   evaluation) into a single unified schema of `(role, text)` turns.
2. **Exploratory data analysis** — label balance, trajectory-length
   distributions (turns and tokens), per-turn length distributions,
   tool-call statistics, and a TF-IDF/SVD lexical projection. Each plot is
   accompanied by an insight that feeds directly into a modelling decision.
3. **Baseline models**, all trained only on benign SWE-smith and scored on
   TRACE and MALT:
   - A token-padded VAE as a first-pass sanity check (expected to fail
     based on the EDA).
   - A cosine-to-centroid baseline on the raw per-turn encoder output
     (no training), which tells us how much the *frozen encoder alone*
     already separates hacked from benign.
   - Two per-turn sentence-embedding VAEs (mean pooling and mean ⊕ max
     pooling), which are compared against the cosine baseline to read off
     how much the VAE actually adds over the frozen encoder.
   - A TF-IDF + SVD + side-features One-Class SVM as a classical
     lexical / structural comparison point.
4. **Final comparison** — AUROC and AUPRC for every model on both
   evaluation sets, with positive-rate baselines called out so the metrics
   stay interpretable on imbalanced data.


### Imports, paths, and global configuration

The configuration block at the top exposes the levers that change runtime and
memory footprint (dataset subsample sizes, encoder model name, VAE epochs).
Defaults target a feasible single-machine run; set limits to `None` for a full
run.

In [ ]:
from __future__ import annotations

import json
import math
import random
import re
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import torch

from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 1090
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# -------- device ----------------------------------------------------------
# Prefer CUDA (Linux/Windows GPUs), then Apple Silicon MPS, then CPU.
if torch.cuda.is_available():
    DEVICE = "cuda"
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
print("DEVICE:", DEVICE)

# -------- paths -----------------------------------------------------------
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CLEAN_DIR = ROOT / "data" / "clean"
SWE_PATH = CLEAN_DIR / "normalized_swe_smith.jsonl"
TRACE_PATH = CLEAN_DIR / "normalized_trace.jsonl"
MALT_PATH = CLEAN_DIR / "normalized_malt.jsonl"

OUT_BASE = ROOT / "results"
FEATURE_DIR = OUT_BASE / "features" / "MS03"
MODEL_DIR = OUT_BASE / "modeling" / "MS03"
TABLE_DIR = OUT_BASE / "tables" / "MS03"
FIG_DIR = OUT_BASE / "figures" / "MS03"
for d in (FEATURE_DIR, MODEL_DIR, TABLE_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

# -------- config ----------------------------------------------------------
# EDA subsample: keep SWE EDA fast but still representative. Set to None for full.
SWE_EDA_LIMIT = 15000
# Training subsample. 5000 is a common canonical size; raise (or set to None)
# for a full-scale run.
SWE_TRAIN_LIMIT = 5000

# Encoder. bge-small-en-v1.5 is the same 384-dim size as all-MiniLM-L6-v2 but
# doubles the per-input context to 512 tokens, which matters because some
# individual tool-output turns are longer than 256 tokens.
ENCODER_MODEL = "BAAI/bge-small-en-v1.5"
# Change to "sentence-transformers/all-MiniLM-L6-v2" (256 tokens) to compare.

# VAE hyperparameters.
LATENT_DIM = 32
HIDDEN_DIM = 256
VAE_EPOCHS = 30
VAE_PATIENCE = 7
VAE_BETA = 1e-3
BATCH_SIZE = 256

# Token-padding sanity check (deliberately small / weak).
TOKEN_MAX_LEN = 512
TOKENIZER_NAME = "distilbert-base-uncased"
TOKEN_VAE_EPOCHS = 10
TOKEN_VAE_PATIENCE = 3
TOKEN_SWE_LIMIT = 5000

print("ROOT:", ROOT)
print("Encoder:", ENCODER_MODEL)
print("SWE_TRAIN_LIMIT:", SWE_TRAIN_LIMIT)


---
## 1. Data Grabbing and Preprocessing

### 1.1 Datasets and unified schema

Three datasets feed the pipeline:

- **SWE-smith trajectories** (training) — multi-turn SWE-agent runs on real
  GitHub issues, all benign. Used only to learn what "normal" agent behaviour
  looks like.
- **TRACE** (evaluation) — a human-verified benchmark of ~517 coding-agent
  trajectories with a roughly balanced split between benign and hacked.
- **MALT** (evaluation) — real-world agent transcripts with manually reviewed
  hacked and benign examples, heavily imbalanced.

Each dataset originally ships in a different format (SWE-agent JSON, ChatML,
METR parquet). A preprocessing pass normalises all three into one on-disk
JSONL schema:

```
{
  "dataset":       "swe_smith" | "trace" | "malt",
  "split":         str,
  "trajectory_id": str,
  "instance_id":   str,
  "binary_label":  0 | 1 | None,        # None for SWE (unlabeled, all benign)
  "raw_label":     original source label (list/int/str),
  "turns":         [ {"role": str, "text": str}, ... ],
  "n_turns":       int,
  "meta":          { ... }
}
```

Tool calls and tool responses are serialised into the `text` field during
normalisation, so every dataset collapses to a simple list of `(role, text)`
turns. That is the primitive we build the rest of the pipeline on top of.

### 1.2 Auto-bootstrap: build normalized data if missing

The loaders below expect three files under `data/clean/`:

```
normalized_swe_smith.jsonl
normalized_trace.jsonl
normalized_malt.jsonl
```

If any of them is missing, this section builds it from scratch: downloading
the raw datasets from HuggingFace (SWE-bench/SWE-smith-trajectories,
PatronusAI/trace-dataset, metr-evals/malt-public) into `data/raw/`, then
normalising each dataset into the unified `{role, text}` schema and writing
the JSONL out. The step is idempotent — already-present files are skipped —
so re-running the notebook is cheap. On a first run this step requires the
`datasets` package and several GB of free disk.

In [ ]:
RAW_DIR = ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

# Optional HuggingFace token for private MALT download. Leave as None for
# public-only access; set via the HF_TOKEN environment variable if needed.
import os
HF_TOKEN = os.environ.get("HF_TOKEN")

HF_DATASETS = {
    "swe_smith_trajectories": "SWE-bench/SWE-smith-trajectories",
    "trace":                  "PatronusAI/trace-dataset",
    "malt":                   "metr-evals/malt-public",
}

# Label sets used by the MALT normalization. Match the upstream preprocessing.
TRACE_BENIGN_LABEL = "0"
MALT_HACK_LABELS = {
    "bypass_constraints", "hardcoded_solution", "ignores_task_instructions",
}
MALT_NON_BENIGN_LABELS = {
    "refusals", "reasoning_about_task", "partial_problem_solving",
    "match_weaker_model", "gives_up", "sabotage",
}


def ensure_raw_downloaded():
    """Download each raw HF dataset into data/raw/<save_name>/<split>/ if missing."""
    from datasets import load_dataset

    for save_name, hf_name in HF_DATASETS.items():
        target_dir = RAW_DIR / save_name
        if target_dir.exists() and any(target_dir.iterdir()):
            continue
        print(f"downloading {hf_name} -> {target_dir}")
        ds = load_dataset(hf_name, token=HF_TOKEN)
        target_dir.mkdir(parents=True, exist_ok=True)
        for split in ds:
            ds[split].save_to_disk(str(target_dir / split))


def _safe_json(x):
    return json.loads(x) if isinstance(x, str) else x


def _stringify(x):
    if x is None:               return ""
    if isinstance(x, str):      return x
    if isinstance(x, (int, float, bool)): return str(x)
    if isinstance(x, list):
        return "\n".join(filter(None, (_stringify(i) for i in x)))
    if isinstance(x, dict):
        if isinstance(x.get("text"), str):
            return x["text"]
        return json.dumps(x, ensure_ascii=False, sort_keys=True)
    return str(x)


def _msg_to_turn(msg):
    pieces = []
    content = _stringify(msg.get("content")).strip()
    if content:
        pieces.append(content)
    for key in ["tool_calls", "tool_results", "function_call", "action",
                "message_type", "name"]:
        val = msg.get(key)
        if val not in (None, "", [], {}):
            pieces.append(f"[{key.upper()}] {_stringify(val)}")
    text = "\n".join(pieces).strip()
    return {"role": msg.get("role", "unknown"), "text": text}


def _normalize_turns(messages):
    return [t for t in (_msg_to_turn(m) for m in messages) if t["text"]]


def _load_raw_dataset(save_name):
    """Load all splits under data/raw/<save_name>/ via load_from_disk."""
    from datasets import load_from_disk, DatasetDict
    base = RAW_DIR / save_name
    splits = {p.name: load_from_disk(str(p)) for p in base.iterdir() if p.is_dir()}
    return DatasetDict(splits)


def _write_normalized_swe(path):
    ds = _load_raw_dataset("swe_smith_trajectories")
    with open(path, "w", encoding="utf-8") as f:
        for split_name, split in ds.items():
            for row in split:
                turns = _normalize_turns(_safe_json(row["messages"]))
                rec = {
                    "dataset": "swe_smith", "split": split_name,
                    "trajectory_id": row.get("traj_id"),
                    "instance_id":   row.get("instance_id"),
                    "binary_label":  None,
                    "raw_label":     None,
                    "turns":         turns,
                    "n_turns":       len(turns),
                    "meta": {
                        "resolved": row.get("resolved"),
                        "model":    row.get("model"),
                        "patch":    row.get("patch"),
                    },
                }
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def _write_normalized_trace(path):
    ds = _load_raw_dataset("trace")
    with open(path, "w", encoding="utf-8") as f:
        for split_name, split in ds.items():
            for row in split:
                turns = _normalize_turns(_safe_json(row["conversation"]))
                raw_label = str(row.get("label")).strip()
                rec = {
                    "dataset": "trace", "split": split_name,
                    "trajectory_id": row.get("trajectory_id"),
                    "instance_id":   None,
                    "binary_label":  0 if raw_label == TRACE_BENIGN_LABEL else 1,
                    "raw_label":     raw_label,
                    "turns":         turns,
                    "n_turns":       len(turns),
                    "meta": {},
                }
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def _choose_longest(samples):
    best, best_len = None, -1
    for s in samples:
        inp = s.get("input", []) or []
        outs = s.get("output", []) or []
        out_msgs = outs[0] if outs and isinstance(outs[0], list) else []
        total = len(inp) + len(out_msgs)
        if total > best_len:
            best, best_len = s, total
    return best


def _write_normalized_malt(path, remove_cot=False):
    ds = _load_raw_dataset("malt")
    with open(path, "w", encoding="utf-8") as f:
        for row in ds["public"]:
            meta = row.get("metadata", {})
            if not meta.get("manually_reviewed", False):
                continue
            if remove_cot and meta.get("has_chain_of_thought", False):
                continue
            labels = set(meta.get("labels", []) or [])
            if labels & MALT_HACK_LABELS:
                binary_label = 1
            elif ("normal" in labels and not (labels & MALT_HACK_LABELS)
                  and not (labels & MALT_NON_BENIGN_LABELS)):
                binary_label = 0
            else:
                continue
            sample = _choose_longest(row.get("samples", []))
            if sample is None:
                continue
            inp = sample.get("input", []) or []
            outs = sample.get("output", []) or []
            out_msgs = outs[0] if outs and isinstance(outs[0], list) else []
            turns = _normalize_turns(inp + out_msgs)
            rec = {
                "dataset": "malt", "split": "public",
                "trajectory_id": f"run_{meta.get('run_id', 'unknown')}",
                "instance_id":   None,
                "binary_label":  binary_label,
                "raw_label":     sorted(labels),
                "turns":         turns,
                "n_turns":       len(turns),
                "meta": {
                    "run_id":   meta.get("run_id"),
                    "task_id":  meta.get("task_id"),
                    "model":    meta.get("model"),
                    "run_source": meta.get("run_source"),
                    "public":   meta.get("public"),
                    "has_chain_of_thought": meta.get("has_chain_of_thought"),
                },
            }
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def ensure_normalized_jsonl():
    needs_download = any(
        not (RAW_DIR / name).exists() or not any((RAW_DIR / name).iterdir())
        for name in HF_DATASETS
    )
    needs_build = any(not p.exists() for p in (SWE_PATH, TRACE_PATH, MALT_PATH))

    if not needs_build:
        print("normalized JSONL files already present -> skipping bootstrap")
        return

    if needs_download:
        ensure_raw_downloaded()

    if not SWE_PATH.exists():
        print("writing", SWE_PATH)
        _write_normalized_swe(SWE_PATH)
    if not TRACE_PATH.exists():
        print("writing", TRACE_PATH)
        _write_normalized_trace(TRACE_PATH)
    if not MALT_PATH.exists():
        print("writing", MALT_PATH)
        _write_normalized_malt(MALT_PATH)


ensure_normalized_jsonl()


### 1.3 Helpers for loading and flattening

In [ ]:
def iter_jsonl(path: Path):
    """Stream a JSONL file line by line."""
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)


def turns_to_text(turns: list[dict]) -> str:
    """Flatten (role, text) turns into a single prefixed string."""
    parts = []
    for t in turns:
        role = t.get("role", "unknown")
        text = t.get("text", "") or ""
        parts.append(f"[{role}] {text}")
    return "\n".join(parts)


def approx_token_count(turns: list[dict]) -> int:
    """Whitespace-split approximation. Cheap proxy for tokenizer token count."""
    return sum(len((t.get("text") or "").split()) for t in turns)


def _record_from_obj(obj, dataset_name):
    turns = obj.get("turns") or []
    if not turns:
        return None
    token_counts = [len((t.get("text") or "").split()) for t in turns]
    raw_label = obj.get("raw_label")
    if isinstance(raw_label, list):
        raw_label = ",".join(map(str, raw_label))
    return {
        "dataset": dataset_name,
        "trajectory_id": str(obj.get("trajectory_id")),
        "binary_label": obj.get("binary_label"),
        "raw_label": raw_label,
        "turns": turns,
        "n_turns": len(turns),
        "approx_tokens": sum(token_counts),
        "first_turn_tokens": token_counts[0],
        "last_turn_tokens": token_counts[-1],
        "mean_tokens_per_turn": float(np.mean(token_counts)),
        "max_tokens_per_turn": int(np.max(token_counts)),
    }


def load_unified(path: Path, dataset_name: str, limit: int | None = None, seed: int = SEED):
    """Load a normalized JSONL into a unified DataFrame.

    If ``limit`` is set, uses Algorithm-R reservoir sampling so only ``limit``
    rows are ever in memory. This matters for SWE-smith (~10 GB on disk).
    """
    rng = random.Random(seed)
    if limit is None:
        records = []
        for obj in iter_jsonl(path):
            rec = _record_from_obj(obj, dataset_name)
            if rec is not None:
                records.append(rec)
    else:
        reservoir = []
        for i, obj in enumerate(iter_jsonl(path)):
            rec = _record_from_obj(obj, dataset_name)
            if rec is None:
                continue
            if len(reservoir) < limit:
                reservoir.append(rec)
            else:
                j = rng.randint(0, i)
                if j < limit:
                    reservoir[j] = rec
        records = reservoir

    df = pd.DataFrame(records).reset_index(drop=True)

    if dataset_name == "swe_smith":
        df["binary_label"] = 0  # All SWE-smith is benign.
    else:
        df["binary_label"] = df["binary_label"].fillna(0).astype(int)

    return df


### 1.4 Load all three datasets

For modeling, SWE-smith is subsampled to `SWE_TRAIN_LIMIT` (default 5000).
EDA uses a larger `SWE_EDA_LIMIT` sample so length distributions are
well-estimated.

In [ ]:
swe_eda_df = load_unified(SWE_PATH, "swe_smith", limit=SWE_EDA_LIMIT)
trace_df = load_unified(TRACE_PATH, "trace", limit=None)
malt_df = load_unified(MALT_PATH, "malt", limit=None)

# Modeling uses the tighter SWE subsample, drawn deterministically as a subset of
# the EDA sample so that all of the modeling rows appear in the EDA distributions.
swe_df = swe_eda_df.sample(
    n=min(SWE_TRAIN_LIMIT, len(swe_eda_df)), random_state=SEED
).reset_index(drop=True)

print("SWE (eda):  ", swe_eda_df.shape)
print("SWE (train):", swe_df.shape)
print("TRACE:      ", trace_df.shape, trace_df['binary_label'].value_counts().to_dict())
print("MALT:       ", malt_df.shape, malt_df['binary_label'].value_counts().to_dict())


### 1.5 Label distribution

In [ ]:
label_counts = (
    pd.concat([swe_eda_df, trace_df, malt_df])
    .groupby(["dataset", "binary_label"]).size().rename("n").reset_index()
)
display(label_counts)

pivot = label_counts.pivot(index="dataset", columns="binary_label", values="n").fillna(0)
pivot.columns = ["benign (0)", "hacked (1)"] if 1 in pivot.columns else ["benign (0)"]
ax = pivot.plot(kind="bar", figsize=(8, 4))
ax.set_ylabel("Number of trajectories")
ax.set_title("Label distribution by dataset")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIG_DIR / "label_counts_by_dataset.png", dpi=200, bbox_inches="tight")
plt.show()


**Insight.** Class balance differs sharply across datasets:

- **SWE-smith**: benign only — no hacked examples. This is the training set.
- **TRACE**: ~balanced (~249 benign, ~268 hacked). AUPRC baseline ≈ 0.52.
- **MALT**: heavily imbalanced (~8% hacked after manual review filtering).
  AUPRC baseline ≈ 0.08; a model must do substantially better than that number
  to be meaningful.

Because we train only on benign data, class imbalance does not affect training.
It matters only for interpreting evaluation metrics, and is the reason we
report AUROC (robust to imbalance) *and* AUPRC (sensitive to how early in the
ranking hacked examples appear).

---
## 2. Exploratory Data Analysis

The EDA here is not decorative: each block feeds a specific downstream
decision. Trajectory lengths tell us what any fixed-length representation
needs to handle, per-turn lengths tell us whether a frozen sentence encoder
can see individual turns without truncation, and tool-call statistics tell us
how much hand-crafted structure is available before we train anything.

### 2.1 Turn counts per dataset × label

In [ ]:
all_df = pd.concat([swe_eda_df, trace_df, malt_df], ignore_index=True)

turn_summary = all_df.groupby(["dataset", "binary_label"])["n_turns"].describe()
display(turn_summary)
turn_summary.to_csv(TABLE_DIR / "turn_count_summary.csv")

fig, ax = plt.subplots(figsize=(9, 4.5))
groups, labels = [], []
for dataset in ["swe_smith", "trace", "malt"]:
    for lbl in sorted(all_df.loc[all_df.dataset == dataset, "binary_label"].unique()):
        sub = all_df[(all_df.dataset == dataset) & (all_df.binary_label == lbl)]["n_turns"]
        if len(sub):
            groups.append(sub.values)
            labels.append(f"{dataset}\n{'hacked' if lbl==1 else 'benign'}")
ax.boxplot(groups, labels=labels, showfliers=False)
ax.set_ylabel("Number of turns")
ax.set_title("Trajectory length (turns) by dataset and label")
plt.tight_layout()
plt.savefig(FIG_DIR / "turn_counts_by_dataset_label_boxplot.png", dpi=200, bbox_inches="tight")
plt.show()


**Insight.** SWE-smith trajectories are long (median ~40–50 turns).
TRACE sits around 25. MALT is bimodal depending on task type. Any fixed-length
representation has to handle this spread gracefully — cheap truncation will
not.

### 2.2 Approximate token counts for the full trajectory

In [ ]:
token_summary = all_df.groupby(["dataset", "binary_label"])["approx_tokens"].describe()
display(token_summary)
token_summary.to_csv(TABLE_DIR / "approx_token_summary.csv")

fig, ax = plt.subplots(figsize=(9, 4.5))
groups, labels = [], []
for dataset in ["swe_smith", "trace", "malt"]:
    for lbl in sorted(all_df.loc[all_df.dataset == dataset, "binary_label"].unique()):
        sub = all_df[(all_df.dataset == dataset) & (all_df.binary_label == lbl)]["approx_tokens"]
        if len(sub):
            groups.append(sub.values)
            labels.append(f"{dataset}\n{'hacked' if lbl==1 else 'benign'}")
ax.boxplot(groups, labels=labels, showfliers=False)
ax.set_ylabel("Approximate tokens (whitespace split)")
ax.set_title("Trajectory length (tokens) by dataset and label")
plt.tight_layout()
plt.savefig(FIG_DIR / "approx_token_counts_by_dataset_label_boxplot.png", dpi=200, bbox_inches="tight")
plt.show()


**Insight — the load-bearing one.** Full trajectories are enormous:
SWE-smith medians sit in the tens of thousands of tokens; TRACE and MALT are
smaller but still routinely exceed 2–5K tokens. Any approach that tokenizes
and pads the *concatenated* trajectory to 512 tokens keeps less than 5% of
the trajectory on average, and essentially none of the late-trajectory turns
where reward-hacking behaviour typically manifests. We will confirm this
empirically in §4.

### 2.3 Per-turn lengths

The per-turn view decides whether a per-turn encoder is viable. Any sentence
encoder has a fixed context window (bge-small = 512 tokens, MiniLM = 256). If
individual turns almost always fit in that window, per-turn encoding avoids
the trajectory-level truncation problem entirely.

In [ ]:
per_turn_stats = all_df[[
    "dataset", "binary_label", "n_turns", "approx_tokens",
    "first_turn_tokens", "last_turn_tokens", "mean_tokens_per_turn", "max_tokens_per_turn",
]].copy()
per_turn_summary = (
    per_turn_stats.groupby(["dataset"])  # pooled over labels for the summary table
    .agg(
        n_trajectories=("n_turns", "size"),
        mean_n_turns=("n_turns", "mean"),
        mean_first_turn_tokens=("first_turn_tokens", "mean"),
        median_first_turn_tokens=("first_turn_tokens", "median"),
        mean_mean_tokens_per_turn=("mean_tokens_per_turn", "mean"),
        mean_max_tokens_per_turn=("max_tokens_per_turn", "mean"),
        median_max_tokens_per_turn=("max_tokens_per_turn", "median"),
        p95_max_tokens_per_turn=("max_tokens_per_turn", lambda s: float(np.percentile(s, 95))),
        p99_max_tokens_per_turn=("max_tokens_per_turn", lambda s: float(np.percentile(s, 99))),
    )
)
display(per_turn_summary)
per_turn_summary.to_csv(TABLE_DIR / "per_turn_length_summary.csv")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, title in zip(
    axes,
    ["first_turn_tokens", "mean_tokens_per_turn", "max_tokens_per_turn"],
    ["First-turn length", "Mean tokens per turn", "Max tokens per turn"],
):
    for dataset, sub in per_turn_stats.groupby("dataset"):
        vals = sub[col].clip(upper=np.percentile(sub[col], 99))
        ax.hist(vals, bins=40, alpha=0.5, label=dataset, density=True)
    ax.axvline(256, color="k", linestyle="--", linewidth=1, label="MiniLM cap (256)")
    ax.axvline(512, color="r", linestyle="--", linewidth=1, label="bge-small cap (512)")
    ax.set_title(title)
    ax.set_xlabel("tokens")
    ax.legend(fontsize=7)
plt.tight_layout()
plt.savefig(FIG_DIR / "per_turn_length_distributions.png", dpi=200, bbox_inches="tight")
plt.show()


**Insight.**

- **First turns** (usually a system/developer prompt or a task description) are
  typically well under 512 tokens for TRACE and MALT; SWE-smith's first turn is
  usually a short system header (often < 50 tokens).
- **Mean tokens per turn** is small (tens to low hundreds) across all datasets.
- **Max tokens per turn** is where the tail matters. A small but non-trivial
  fraction of turns (typically tool outputs — large `stderr`, long file dumps)
  exceed 512 tokens. Under bge-small's 512-token window, those turns still get
  truncated, but only at the *turn* level, not at the trajectory level. The
  comparison with MiniLM's 256-token cap shows why we prefer bge-small.

This is the empirical justification for per-turn encoding: individual turns
mostly fit, full trajectories do not.

### 2.4 Tool-call statistics

Whether hacked vs. benign trajectories use tools at different rates is a
reusable hand-crafted feature for later baselines.

In [ ]:
TOOL_PATTERNS = {
    "bash_calls": re.compile(r"\b(bash|run_bash|shell|terminal)\b", re.I),
    "file_write_calls": re.compile(r"\b(write|create|str_replace|apply_patch|edit)\b", re.I),
    "test_calls": re.compile(r"\b(pytest|unittest|npm test|cargo test|\btest\b)\b", re.I),
}

def count_tool_calls(turns):
    text = "\n".join((t.get("text") or "") for t in turns)
    return {name: len(pat.findall(text)) for name, pat in TOOL_PATTERNS.items()}

tool_rows = []
for df_, name in [(swe_eda_df, "swe_smith"), (trace_df, "trace"), (malt_df, "malt")]:
    for _, row in df_.iterrows():
        rec = count_tool_calls(row["turns"])
        rec.update({"dataset": name, "binary_label": row["binary_label"]})
        tool_rows.append(rec)
tool_df = pd.DataFrame(tool_rows)

tool_summary = tool_df.groupby(["dataset", "binary_label"]).mean(numeric_only=True)
display(tool_summary)
tool_summary.to_csv(TABLE_DIR / "tool_call_summary.csv")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ["bash_calls", "file_write_calls", "test_calls"]):
    grouped = tool_df.groupby(["dataset", "binary_label"])[col].mean().unstack()
    grouped.plot(kind="bar", ax=ax)
    ax.set_title(f"Mean {col} per trajectory")
    ax.set_ylabel(col)
    plt.setp(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.savefig(FIG_DIR / "tool_call_statistics.png", dpi=200, bbox_inches="tight")
plt.show()


**Insight.** Hacked trajectories in TRACE show elevated rates of file
writes and test executions — consistent with the "modify tests to pass" family
of hacks. MALT's hacks are more varied (`bypass_constraints`,
`hardcoded_solution`), and tool-call frequency alone is less discriminative
there. This is why we need a learned representation and not just hand-crafted
counts.

### 2.5 Lexical visualization: TF-IDF + SVD 2D scatter

A classical lexical sanity check: project a balanced sample of trajectories
into a shared 2D space built from TF-IDF features, and colour by dataset and
label. This tells us whether the three datasets occupy distinct regions of
lexical space (domain shift) or whether the hacked/benign axis is already
visible at the surface-text level.

In [ ]:
def _concat_text(turns):
    return "\n".join(f"[{t.get('role','unknown')}] {t.get('text','') or ''}" for t in turns)


def _balanced_sample(df, per_label_limit, seed=SEED):
    pieces = []
    for lbl, sub in df.groupby("binary_label"):
        pieces.append(sub.sample(n=min(len(sub), per_label_limit), random_state=seed))
    return pd.concat(pieces, ignore_index=True)


viz_swe   = swe_eda_df.sample(n=min(300, len(swe_eda_df)), random_state=SEED)
viz_trace = _balanced_sample(trace_df, per_label_limit=100)
viz_malt  = _balanced_sample(malt_df,  per_label_limit=100)

viz_df = pd.concat([viz_swe, viz_trace, viz_malt], ignore_index=True)
viz_df["text"] = viz_df["turns"].apply(_concat_text)
print(viz_df.groupby(["dataset", "binary_label"]).size())

vectorizer_viz = TfidfVectorizer(max_features=5000, stop_words="english", min_df=2)
X_viz = vectorizer_viz.fit_transform(viz_df["text"])
svd_viz = TruncatedSVD(n_components=2, random_state=SEED)
coords = svd_viz.fit_transform(X_viz)
viz_df["x"], viz_df["y"] = coords[:, 0], coords[:, 1]

plt.figure(figsize=(8, 6))
for label, mask in [
    ("SWE-smith train", viz_df["dataset"] == "swe_smith"),
    ("TRACE benign",    (viz_df["dataset"] == "trace") & (viz_df["binary_label"] == 0)),
    ("TRACE hacked",    (viz_df["dataset"] == "trace") & (viz_df["binary_label"] == 1)),
    ("MALT benign",     (viz_df["dataset"] == "malt")  & (viz_df["binary_label"] == 0)),
    ("MALT hacked",     (viz_df["dataset"] == "malt")  & (viz_df["binary_label"] == 1)),
]:
    if mask.sum() == 0:
        continue
    plt.scatter(viz_df.loc[mask, "x"], viz_df.loc[mask, "y"], alpha=0.6, label=label)
plt.title("Trajectory text visualization (TF-IDF + SVD)")
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / "trajectory_text_visualization_tfidf_svd.png", dpi=200, bbox_inches="tight")
plt.show()


**Insight.** The three datasets typically cluster by *source* rather than by
*label*. SWE-smith, TRACE, and MALT occupy distinct regions of lexical space;
hacked vs. benign points within each evaluation dataset overlap much more
than any across-dataset pair does. Practical consequence: any unsupervised
detector trained only on SWE-smith will flag a lot of TRACE/MALT examples as
anomalous simply because they *look different from SWE-smith*, regardless of
whether they are hacked. We have to interpret every downstream anomaly score
with domain shift in mind.

### 2.6 Key EDA insights (summary)

1. **Trajectory length varies dramatically** across datasets (medians of ~25
   to 50+ turns and thousands to tens of thousands of tokens). Any
   representation must be length-invariant.
2. **Full-trajectory truncation is catastrophic** at 512 tokens — a median
   SWE trajectory contains 20–40× that. A padded-sequence model operating on
   the concatenated trajectory will only ever see the opening frames.
3. **Per-turn lengths fit comfortably** in a 512-token encoder window for
   the vast majority of turns. This motivates per-turn encoding followed by
   pooling rather than trajectory-level tokenization.
4. **Tool-call patterns differ** between hacked and benign on TRACE but are
   less clean on MALT, so we cannot rely on hand-crafted action counts alone.
5. **Datasets cluster by source rather than by label** in the TF-IDF/SVD
   projection. Any unsupervised detector trained only on SWE-smith risks
   picking up domain shift as much as hacking. Positive-rate baselines on
   each evaluation set are the guardrail.

---
## 3. Representation Strategy

Four candidate representations, ranked by how seriously we take them:

| Rep | Description | Role |
|---|---|---|
| A | Padded token IDs of the concatenated trajectory (max_len=512) | Sanity check — expected to fail based on the EDA above |
| B | Per-turn embeddings, **mean-pooled** | Primary learned baseline |
| C | Per-turn embeddings, **mean ⊕ max concat** | Second aggregation — tests whether peak-per-dim signal adds information |
| D | TF-IDF + SVD + side features | Lexical / structural comparison point |

Plus a **cosine-to-centroid baseline** in the raw encoder space (no training)
to check whether the frozen encoder already separates hacked from benign.

**Encoder choice (for Reps B and C).** We use `BAAI/bge-small-en-v1.5`:

- 384-dim output — small and fast, same footprint as MiniLM-L6-v2.
- 512-token input window, twice MiniLM's 256 — safer for the occasional
  long tool-output turn.
- Strong general-purpose sentence embeddings; widely used in retrieval, so
  the vector space is reasonably well-behaved without fine-tuning.

We keep the encoder **frozen** throughout. The VAE is a small MLP on top of
the pooled vector.

**Model choice.** Reps B and C feed a benign-only VAE. Rep D feeds a
benign-only One-Class SVM, since TF-IDF + SVD is a classical lexical
representation for which a kernel/linear SVM is the natural match. This
gives us *two different unsupervised model families* on *different
representations*, so we can separate "representation helps" from "model
helps" in the final comparison.

---
## 4. Sanity Check — Token-Padding VAE

**Hypothesis (from EDA).** Padding/truncating the concatenated trajectory to
512 tokens discards ≥95% of the content on average, so a VAE trained on that
representation should not produce meaningful anomaly scores.

Before investing in the per-turn encoder pipeline, we run exactly this
baseline once to make the failure mode explicit and numerical. AUROC near
0.5 and AUPRC near the dataset positive rate would confirm the hypothesis
and rule out token-level concatenation as a representation. If it
unexpectedly worked, that would itself be informative.

In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
PAD_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
VOCAB_SIZE = tokenizer.vocab_size

def flatten_texts(df):
    return df["turns"].apply(turns_to_text).tolist()

def tokenize_block(texts, batch_size=128):
    ids_chunks, mask_chunks = [], []
    for start in range(0, len(texts), batch_size):
        enc = tokenizer(
            texts[start:start + batch_size],
            padding="max_length", truncation=True,
            max_length=TOKEN_MAX_LEN, return_attention_mask=True,
        )
        ids_chunks.append(np.asarray(enc["input_ids"], dtype=np.int32))
        mask_chunks.append(np.asarray(enc["attention_mask"], dtype=np.int8))
    return np.vstack(ids_chunks), np.vstack(mask_chunks)

def tokenize_cached(df, name):
    path = FEATURE_DIR / f"tokens_{name}_n{len(df)}_maxlen{TOKEN_MAX_LEN}.npz"
    if path.exists():
        arr = np.load(path)
        return arr["input_ids"], arr["attention_mask"]
    ids, mask = tokenize_block(flatten_texts(df))
    np.savez_compressed(path, input_ids=ids, attention_mask=mask)
    return ids, mask

swe_token_df = swe_df.sample(
    n=min(TOKEN_SWE_LIMIT, len(swe_df)), random_state=SEED
).reset_index(drop=True)

swe_ids, swe_mask   = tokenize_cached(swe_token_df, "swe")
trace_ids, trace_mask = tokenize_cached(trace_df, "trace")
malt_ids, malt_mask   = tokenize_cached(malt_df, "malt")

# EDA-driven diagnostic: what fraction of tokens is truncated?
full_wstoks = swe_token_df["approx_tokens"].values  # whitespace tokens
retained_frac = np.clip(swe_mask.sum(axis=1) / np.maximum(full_wstoks, 1), 0, 1)
print(f"SWE median retained token fraction at max_len={TOKEN_MAX_LEN}: "
      f"{np.median(retained_frac):.3f}")
print(f"SWE 90th-percentile retained token fraction:                 "
      f"{np.percentile(retained_frac, 90):.3f}")


In [ ]:
class TokenPadVAE(nn.Module):
    """Minimal VAE operating on masked-mean-pooled token embeddings."""
    def __init__(self, vocab_size, max_len, emb_dim=64, latent_dim=32,
                 hidden_dim=256, pad_id=0):
        super().__init__()
        self.max_len = max_len
        self.emb_dim = emb_dim
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.encoder = nn.Sequential(
            nn.Linear(emb_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(),
        )
        self.mu_head = nn.Linear(hidden_dim // 2, latent_dim)
        self.lv_head = nn.Linear(hidden_dim // 2, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, emb_dim),
        )

    def pool(self, emb, mask):
        m = mask.unsqueeze(-1).float()
        return (emb * m).sum(dim=1) / m.sum(dim=1).clamp(min=1.0)

    def forward(self, ids, mask):
        emb = self.embedding(ids)
        pooled = self.pool(emb, mask)
        h = self.encoder(pooled)
        mu, logvar = self.mu_head(h), self.lv_head(h)
        std = (0.5 * logvar).exp()
        z = mu + torch.randn_like(std) * std
        recon = self.decoder(z)
        return recon, pooled.detach(), mu, logvar


def token_vae_loss(recon, target, mu, logvar, beta=VAE_BETA):
    recon_loss = ((recon - target) ** 2).mean(dim=1)
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)
    score = recon_loss + beta * kl
    return score.mean(), recon_loss, kl, score


def score_token_vae(model, ids, mask, batch_size=128):
    ds = TensorDataset(torch.tensor(ids, dtype=torch.long),
                       torch.tensor(mask, dtype=torch.long))
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
    scores = []
    model.eval()
    with torch.no_grad():
        for i, m in loader:
            i, m = i.to(DEVICE), m.to(DEVICE)
            recon, target, mu, logvar = model(i, m)
            _, _, _, s = token_vae_loss(recon, target, mu, logvar)
            scores.extend(s.cpu().numpy())
    return np.asarray(scores)


def train_token_vae(ids, mask, epochs=TOKEN_VAE_EPOCHS, patience=TOKEN_VAE_PATIENCE):
    train_idx, val_idx = train_test_split(
        np.arange(len(ids)), test_size=0.2, random_state=SEED
    )
    train_ids, val_ids = ids[train_idx], ids[val_idx]
    train_mask, val_mask = mask[train_idx], mask[val_idx]

    ds = TensorDataset(torch.tensor(train_ids, dtype=torch.long),
                       torch.tensor(train_mask, dtype=torch.long))
    loader = DataLoader(ds, batch_size=128, shuffle=True)

    model = TokenPadVAE(
        vocab_size=VOCAB_SIZE, max_len=TOKEN_MAX_LEN,
        emb_dim=64, latent_dim=LATENT_DIM, hidden_dim=HIDDEN_DIM, pad_id=PAD_ID,
    ).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    best_val, best_state, waited = math.inf, None, 0
    for epoch in range(1, epochs + 1):
        model.train()
        for i, m in loader:
            i, m = i.to(DEVICE), m.to(DEVICE)
            opt.zero_grad()
            recon, target, mu, logvar = model(i, m)
            loss, _, _, _ = token_vae_loss(recon, target, mu, logvar)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        val_scores = score_token_vae(model, val_ids, val_mask)
        val_loss = float(val_scores.mean())
        print(f"epoch {epoch:02d}  val_loss={val_loss:.5f}")
        if val_loss < best_val - 1e-6:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            waited = 0
        else:
            waited += 1
            if waited >= patience:
                print("early stopping")
                break
    model.load_state_dict(best_state)
    return model


In [ ]:
tokpad_model = train_token_vae(swe_ids, swe_mask)


### 4.1 Scoring TRACE and MALT

In [ ]:
def metrics_from_scores(y_true, score):
    y_true = np.asarray(y_true).astype(int)
    score = np.asarray(score).astype(float)
    out = {"n": len(y_true), "positive_rate": float(y_true.mean()),
           "auroc": np.nan, "auprc": np.nan}
    if len(np.unique(y_true)) > 1:
        out["auroc"] = float(roc_auc_score(y_true, score))
        out["auprc"] = float(average_precision_score(y_true, score))
    return out


tokpad_results = []
for dname, df_, ids_, mask_ in [
    ("trace", trace_df, trace_ids, trace_mask),
    ("malt",  malt_df,  malt_ids,  malt_mask),
]:
    s = score_token_vae(tokpad_model, ids_, mask_)
    tokpad_results.append({
        "model": "vae",
        "representation": "tokpad_512 (sanity check)",
        "dataset": dname,
        **metrics_from_scores(df_["binary_label"].values, s),
    })
tokpad_results = pd.DataFrame(tokpad_results)
display(tokpad_results)
tokpad_results.to_csv(TABLE_DIR / "sanity_tokpad_results.csv", index=False)


**Sanity-check insight.** AUROC near 0.5 and AUPRC near the positive rate
confirms the EDA story: when the representation only sees the first ~5% of
the trajectory, the model cannot see the hack. Any residual signal it picks
up is driven by opening-prompt style differences between datasets (domain
shift), not by hacking.

**Conclusion.** Drop token-padding as a serious representation. Everything
that follows uses per-turn embeddings.

---
## 5. Primary Representation — Per-Turn Sentence Embeddings

Each turn is encoded independently with a frozen sentence encoder, so each
turn keeps its own semantic vector. We then aggregate turn vectors into a
single trajectory vector. Two aggregations:

- **mean**: simple average over turns. Captures "typical" behaviour of the
  trajectory.
- **mean ⊕ max**: concatenation of element-wise mean and element-wise max.
  Adds per-dimension peak activation — the intuition is that reward-hacking
  moments may be localised to a handful of turns that a mean dilutes away.

### 5.1 Encoder loading and per-turn encoding

Each turn is encoded independently with `BAAI/bge-small-en-v1.5`. The encoder
is kept **frozen** — its weights are never updated. Turn-level vectors are
cached to disk so re-running this section is cheap. The encoder requires
`sentence-transformers` to be installed in the environment.

In [ ]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer(ENCODER_MODEL)
try:
    encoder = encoder.to(DEVICE)
except Exception:
    pass
EMB_DIM = encoder.get_sentence_embedding_dimension()
print("Encoder:", ENCODER_MODEL, "| dim:", EMB_DIM)


def encode_turns_cached(df, name, batch_size=64):
    """Encode every turn of every trajectory, cached per (name, n_rows).

    Returns a list of np.ndarray, one per trajectory, shape (n_turns, EMB_DIM).
    """
    cache_path = FEATURE_DIR / f"turn_emb_{name}_n{len(df)}_{Path(ENCODER_MODEL).name}.npz"
    if cache_path.exists():
        with np.load(cache_path, allow_pickle=True) as z:
            return [z[f"arr_{i}"] for i in range(len(df))]

    results = []
    for _, row in df.iterrows():
        turn_texts = [((t.get("text") or "").strip() or " ") for t in row["turns"]]
        vecs = encoder.encode(
            turn_texts, batch_size=batch_size, show_progress_bar=False,
            normalize_embeddings=True, convert_to_numpy=True,
        ).astype(np.float32)
        results.append(vecs)

    np.savez_compressed(cache_path, **{f"arr_{i}": v for i, v in enumerate(results)})
    return results


swe_turn_embs   = encode_turns_cached(swe_df,   "swe")
trace_turn_embs = encode_turns_cached(trace_df, "trace")
malt_turn_embs  = encode_turns_cached(malt_df,  "malt")

print("swe:", len(swe_turn_embs), "| example shape:", swe_turn_embs[0].shape)


### 5.2 Aggregations

In [ ]:
def aggregate_mean(turn_vecs_list):
    return np.stack([v.mean(axis=0) for v in turn_vecs_list]).astype(np.float32)


def aggregate_mean_max(turn_vecs_list):
    mean = np.stack([v.mean(axis=0) for v in turn_vecs_list])
    mx   = np.stack([v.max(axis=0)  for v in turn_vecs_list])
    return np.concatenate([mean, mx], axis=1).astype(np.float32)


# Side features: n_turns and trajectory-level token count. We scale them later
# with the rest of the representation so they do not dominate the VAE loss.
def side_features(df):
    return df[["n_turns", "approx_tokens", "mean_tokens_per_turn",
               "max_tokens_per_turn", "first_turn_tokens"]].to_numpy().astype(np.float32)


X_mean = {
    "swe":   aggregate_mean(swe_turn_embs),
    "trace": aggregate_mean(trace_turn_embs),
    "malt":  aggregate_mean(malt_turn_embs),
}

X_meanmax = {
    "swe":   aggregate_mean_max(swe_turn_embs),
    "trace": aggregate_mean_max(trace_turn_embs),
    "malt":  aggregate_mean_max(malt_turn_embs),
}

side = {
    "swe":   side_features(swe_df),
    "trace": side_features(trace_df),
    "malt":  side_features(malt_df),
}


def concat_side(X_dict, side_dict):
    # Fit scaler on SWE side features only (train-only fit).
    scaler = StandardScaler().fit(side_dict["swe"])
    out = {}
    for k in X_dict:
        out[k] = np.hstack([X_dict[k], scaler.transform(side_dict[k]).astype(np.float32)])
    return out, scaler


X_mean_full,    side_scaler_mean    = concat_side(X_mean, side)
X_meanmax_full, side_scaler_meanmax = concat_side(X_meanmax, side)

print("mean rep dims:     ", X_mean_full["swe"].shape[1])
print("mean+max rep dims: ", X_meanmax_full["swe"].shape[1])


### 5.3 Cosine-to-centroid baseline (no training)

A cheap sanity check on whether the frozen encoder already separates hacked from benign. If this baseline is strong, a more expensive VAE is barely adding value; if it is weak and the VAE beats it, training on SWE is earning its keep. This is the simplest possible learned-space baseline, so it goes before the VAE.

In [ ]:
def cosine_to_centroid_baseline(X_dict, name):
    centroid = X_dict["swe"].mean(axis=0)
    centroid_norm = centroid / (np.linalg.norm(centroid) + 1e-12)
    rows = []
    for dname, df_ in [("trace", trace_df), ("malt", malt_df)]:
        V = X_dict[dname]
        V_norm = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-12)
        cos_sim = V_norm @ centroid_norm
        score = 1.0 - cos_sim   # higher = more anomalous
        rows.append({
            "model": "cosine_centroid",
            "representation": name,
            "dataset": dname,
            **metrics_from_scores(df_["binary_label"].values, score),
        })
    return pd.DataFrame(rows)


# Note: baseline runs on the embedding matrix *without* side features for a
# pure "frozen encoder" read.
cosine_mean_results    = cosine_to_centroid_baseline(X_mean,    "turn_emb_mean")
cosine_meanmax_results = cosine_to_centroid_baseline(X_meanmax, "turn_emb_mean_max")

display(pd.concat([cosine_mean_results, cosine_meanmax_results], ignore_index=True))


### 5.4 VAE on the aggregated representation

In [ ]:
class VectorVAE(nn.Module):
    def __init__(self, input_dim, latent_dim=LATENT_DIM, hidden_dim=HIDDEN_DIM):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(),
        )
        self.mu_head = nn.Linear(hidden_dim // 2, latent_dim)
        self.lv_head = nn.Linear(hidden_dim // 2, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim // 2), nn.ReLU(),
            nn.Linear(hidden_dim // 2, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
        )

    def forward(self, x):
        h = self.encoder(x)
        mu, logvar = self.mu_head(h), self.lv_head(h)
        std = (0.5 * logvar).exp()
        z = mu + torch.randn_like(std) * std
        return self.decoder(z), mu, logvar


def vae_loss(recon, x, mu, logvar, beta=VAE_BETA):
    recon_loss = ((recon - x) ** 2).mean(dim=1)
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)
    score = recon_loss + beta * kl
    return score.mean(), recon_loss, kl, score


def train_vector_vae(X_swe, epochs=VAE_EPOCHS, patience=VAE_PATIENCE,
                    batch_size=BATCH_SIZE):
    X_tr_raw, X_val_raw = train_test_split(X_swe, test_size=0.2, random_state=SEED)
    scaler = StandardScaler().fit(X_tr_raw)
    X_tr = scaler.transform(X_tr_raw).astype(np.float32)
    X_val = scaler.transform(X_val_raw).astype(np.float32)

    loader = DataLoader(
        TensorDataset(torch.tensor(X_tr)),
        batch_size=batch_size, shuffle=True,
    )
    val_tensor = torch.tensor(X_val, device=DEVICE)

    model = VectorVAE(input_dim=X_tr.shape[1]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    best_val, best_state, waited = math.inf, None, 0
    for epoch in range(1, epochs + 1):
        model.train()
        for (xb,) in loader:
            xb = xb.to(DEVICE)
            opt.zero_grad()
            recon, mu, logvar = model(xb)
            loss, _, _, _ = vae_loss(recon, xb, mu, logvar)
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            recon, mu, logvar = model(val_tensor)
            val_score = ((recon - val_tensor) ** 2).mean(dim=1).cpu().numpy()
        val_loss = float(val_score.mean())
        if epoch == 1 or epoch % 5 == 0 or waited >= patience - 1:
            print(f"epoch {epoch:02d}  val_mse={val_loss:.5f}")
        if val_loss < best_val - 1e-6:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            waited = 0
        else:
            waited += 1
            if waited >= patience:
                print("early stopping")
                break
    model.load_state_dict(best_state)

    # Thresholds on validation scores (for later alert-style reporting).
    val_score_final = score_vector_vae(model, scaler, X_val_raw)
    thresholds = {
        "alert_threshold_90": float(np.quantile(val_score_final, 0.90)),
        "alert_threshold_95": float(np.quantile(val_score_final, 0.95)),
        "alert_threshold_99": float(np.quantile(val_score_final, 0.99)),
    }
    return model, scaler, thresholds


def score_vector_vae(model, scaler, X):
    X_scaled = scaler.transform(X).astype(np.float32)
    loader = DataLoader(TensorDataset(torch.tensor(X_scaled)), batch_size=512, shuffle=False)
    scores = []
    model.eval()
    with torch.no_grad():
        for (xb,) in loader:
            xb = xb.to(DEVICE)
            recon, mu, logvar = model(xb)
            recon_score = ((recon - xb) ** 2).mean(dim=1)
            kl_score = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)
            scores.extend((recon_score + VAE_BETA * kl_score).cpu().numpy())
    return np.asarray(scores)


In [ ]:
def run_vae_on_rep(rep_name, X_rep):
    print(f"\n=== VAE on {rep_name} ===")
    model, scaler, thresholds = train_vector_vae(X_rep["swe"])
    rows = []
    for dname, df_ in [("trace", trace_df), ("malt", malt_df)]:
        s = score_vector_vae(model, scaler, X_rep[dname])
        rows.append({
            "model": "vae",
            "representation": rep_name,
            "dataset": dname,
            **metrics_from_scores(df_["binary_label"].values, s),
            **thresholds,
        })
    return model, scaler, pd.DataFrame(rows)


vae_mean_model,    vae_mean_scaler,    vae_mean_results    = run_vae_on_rep("turn_emb_mean",     X_mean_full)
vae_meanmax_model, vae_meanmax_scaler, vae_meanmax_results = run_vae_on_rep("turn_emb_mean_max", X_meanmax_full)

display(pd.concat([vae_mean_results, vae_meanmax_results], ignore_index=True))


### 5.5 TF-IDF + SVD + side features — One-Class SVM

A classical lexical representation as a comparison point to the
sentence-embedding baselines. The idea is to see whether surface-level
lexical patterns (plus a handful of hand-crafted structural features)
already carry enough signal to rank hacked trajectories higher than benign
ones, without any neural encoder or VAE.

**Pipeline.**

1. Fit a `TfidfVectorizer` on flattened SWE-smith trajectory text (train-only
   fit, prevents lexical leakage from the evaluation sets).
2. Compress the high-dimensional sparse TF-IDF matrix with `TruncatedSVD` to
   a dense 128-dim vector. SVD is the standard dimensionality reducer for
   sparse text features.
3. Append a small block of **side features** that the bag-of-words cannot
   express on its own: `n_turns`, `approx_tokens`, `mean_tokens_per_turn`,
   `max_tokens_per_turn`, `first_turn_tokens`, and the three tool-call
   counts.
4. Train a `SGDOneClassSVM` on SWE-smith only. At scoring time,
   `-decision_function(x)` is the anomaly score.

In [ ]:
from sklearn.linear_model import SGDOneClassSVM
from sklearn.pipeline import make_pipeline


def flatten_trajectory_text(df):
    return df["turns"].apply(_concat_text).tolist()


def make_structural_side(df, tool_df_subset):
    """Length and tool-call side features aligned with df row order."""
    base = df[[
        "n_turns", "approx_tokens", "mean_tokens_per_turn",
        "max_tokens_per_turn", "first_turn_tokens",
    ]].to_numpy(dtype=np.float32)
    tool_cols = ["bash_calls", "file_write_calls", "test_calls"]
    tools = tool_df_subset[tool_cols].to_numpy(dtype=np.float32)
    return np.hstack([base, tools])


def build_tfidf_svd_rep(svd_dim=128, max_features=50000):
    vec = TfidfVectorizer(
        max_features=max_features, min_df=2, ngram_range=(1, 2),
        lowercase=True, strip_accents="unicode", sublinear_tf=True,
    )
    X_swe_tfidf   = vec.fit_transform(flatten_trajectory_text(swe_df))
    X_trace_tfidf = vec.transform(flatten_trajectory_text(trace_df))
    X_malt_tfidf  = vec.transform(flatten_trajectory_text(malt_df))

    svd = TruncatedSVD(n_components=min(svd_dim, X_swe_tfidf.shape[1] - 1),
                       random_state=SEED)
    X_swe   = svd.fit_transform(X_swe_tfidf).astype(np.float32)
    X_trace = svd.transform(X_trace_tfidf).astype(np.float32)
    X_malt  = svd.transform(X_malt_tfidf).astype(np.float32)
    return (X_swe, X_trace, X_malt), {
        "vocab_size": len(vec.vocabulary_),
        "svd_explained_variance_ratio_sum": float(svd.explained_variance_ratio_.sum()),
    }


# Align tool-call rows back to each modeling dataframe by index.
tool_df_indexed = tool_df.copy()
# tool_df was built in §2.4 in the order (swe_eda_df, trace_df, malt_df),
# but swe_df is a subsample of swe_eda_df. Rebuild a per-row tool-call
# dataframe keyed by trajectory_id so we can join onto swe_df, trace_df, malt_df.
tool_by_source = {}
offset = 0
for src_df, name in [(swe_eda_df, "swe_smith"), (trace_df, "trace"), (malt_df, "malt")]:
    block = tool_df.iloc[offset: offset + len(src_df)].reset_index(drop=True)
    block["trajectory_id"] = src_df["trajectory_id"].values
    tool_by_source[name] = block
    offset += len(src_df)


def tool_rows_for(df, source_name):
    block = tool_by_source[source_name].set_index("trajectory_id")
    return block.reindex(df["trajectory_id"].values).reset_index(drop=True)


(tfidf_swe, tfidf_trace, tfidf_malt), tfidf_meta = build_tfidf_svd_rep()
print("TF-IDF/SVD meta:", tfidf_meta)

side_swe_s   = make_structural_side(swe_df,   tool_rows_for(swe_df,   "swe_smith"))
side_trace_s = make_structural_side(trace_df, tool_rows_for(trace_df, "trace"))
side_malt_s  = make_structural_side(malt_df,  tool_rows_for(malt_df,  "malt"))

side_scaler_svm = StandardScaler().fit(side_swe_s)

X_tfidf_side = {
    "swe":   np.hstack([tfidf_swe,   side_scaler_svm.transform(side_swe_s)  ]).astype(np.float32),
    "trace": np.hstack([tfidf_trace, side_scaler_svm.transform(side_trace_s)]).astype(np.float32),
    "malt":  np.hstack([tfidf_malt,  side_scaler_svm.transform(side_malt_s) ]).astype(np.float32),
}
print("TF-IDF + SVD + side dims:", X_tfidf_side["swe"].shape[1])


In [ ]:
def run_oneclass_svm(X_dict, representation):
    model = make_pipeline(
        StandardScaler(),
        SGDOneClassSVM(nu=0.05, random_state=SEED, max_iter=2000, tol=1e-3),
    )
    model.fit(X_dict["swe"])

    rows = []
    for dname, df_ in [("trace", trace_df), ("malt", malt_df)]:
        score = -model.decision_function(X_dict[dname])
        rows.append({
            "model": "one_class_svm",
            "representation": representation,
            "dataset": dname,
            **metrics_from_scores(df_["binary_label"].values, score),
        })
    return model, pd.DataFrame(rows)


svm_tfidf_model, svm_tfidf_results = run_oneclass_svm(
    X_tfidf_side, representation="tfidf_svd_side",
)
display(svm_tfidf_results)


---
## 6. Final Comparison

All four models in one table.

In [ ]:
final_results = pd.concat([
    tokpad_results,
    cosine_mean_results,
    cosine_meanmax_results,
    vae_mean_results,
    vae_meanmax_results,
    svm_tfidf_results,
], ignore_index=True)

final_results = final_results[[
    "model", "representation", "dataset", "n", "positive_rate", "auroc", "auprc",
] + [c for c in final_results.columns if c.startswith("alert_threshold")]]

final_results = final_results.sort_values(["dataset", "auprc"], ascending=[True, False]).reset_index(drop=True)
final_results.to_csv(TABLE_DIR / "final_results.csv", index=False)
display(final_results)


In [ ]:
for metric in ["auprc", "auroc"]:
    fig, ax = plt.subplots(figsize=(10, max(4, 0.45 * len(final_results))))
    df_plot = final_results.copy()
    df_plot["label"] = (df_plot["dataset"].str.upper() + " | "
                        + df_plot["model"] + " | " + df_plot["representation"])
    df_plot = df_plot.sort_values(metric, ascending=False)
    ax.barh(df_plot["label"], df_plot[metric])
    ax.invert_yaxis()
    ax.set_xlabel(metric.upper())
    ax.set_title(f"MS03 final comparison — {metric.upper()}")

    # Add positive-rate baselines.
    for dname, sub in df_plot.groupby("dataset"):
        ax.axvline(sub["positive_rate"].mean(), linestyle="--", linewidth=0.8, alpha=0.4)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"final_comparison_{metric}.png", dpi=200, bbox_inches="tight")
    plt.show()


### 6.1 How to read the comparison

- **Token-padded VAE (sanity check).** We expect AUROC ≈ 0.5 and AUPRC ≈ the
  dataset positive rate. That result confirms that dropping 95%+ of the
  trajectory removes all hacking signal and rules out token-level
  concatenation as a representation.
- **Cosine-to-centroid baseline (no training).** Measures how much the
  frozen bge-small embedding space *already* separates hacked from benign
  without any training. This is the floor that any learned model has to
  beat.
- **VAE (mean-pool).** The primary learned representation. The delta
  against cosine is the value that training on SWE actually adds.
- **VAE (mean ⊕ max).** Tests whether retaining per-dimension peak
  activations on top of the mean helps. Because hacks are often localised
  to a handful of suspicious turns, the biggest gain should appear on MALT.
- **One-Class SVM on TF-IDF + SVD + side features.** A classical lexical
  comparison point. If it is competitive with or ahead of the VAE
  baselines, the semantic encoder is not earning its weight on this data
  and we should reconsider the representation; if it is clearly behind,
  the per-turn embedding is doing useful work beyond surface lexicon.

Reading the metrics:

- **TRACE is roughly balanced**, so AUPRC ≈ 0.5 is the random-baseline
  regime. Look at AUROC and AUPRC-above-baseline to judge TRACE separation.
- **MALT has ~8% hacked**, so AUPRC > 0.08 is the real bar. Beating it by
  a meaningful margin is the claim we can make.


### 6.2 Limitations and next steps

- **Domain shift is the biggest confound.** Thresholds derived from SWE
  validation reflect benign-SWE variability; TRACE and MALT have different
  scaffolds, tool schemas, and prompts. A high anomaly score may be a
  "different dataset" signal rather than a "hack" signal. A natural next
  step is a per-dataset benign control arm to calibrate domain shift
  explicitly.
- **Aggregation loses order.** Mean and max ignore turn ordering. A sequence
  model (GRU / Transformer) over the per-turn embedding sequence is the
  obvious follow-up and would not require re-encoding, since the per-turn
  vectors are already cached on disk.
- **Deep SVDD.** A natural counterpart to the VAE is Deep SVDD, which
  replaces reconstruction with a compact-hypersphere objective. It is
  expected to be more robust to surface-plausible hacks where reconstruction
  error is low but the trajectory is still structurally anomalous. This is
  out of scope for this notebook.
- **Cross-dataset consistency.** Whether hacked trajectories from TRACE and
  MALT cluster together in the learned latent space is a concrete
  consistency check on the representation and a natural extension of the
  results above.